UN General Debate Transcripts Dataset from *Understanding State Preferences With Text As Data: Introducing the UN General Debate Corpus*

https://journals.sagepub.com/doi/epub/10.1177/2053168017712821

In this notebook, we generate temporal topic models of the UNGDC with various clustering and Mapper parameters.

Main packages: [temporal mapper](https://github.com/TutteInstitute/temporal-mapper) + [toponymy](https://github.com/TutteInstitute/toponymy).

The next cell contains all relevant parameters to be modified.


In [1]:
"""
Parameters
"""
# Set this to agree with `UNGDC-DataPrep.ipynb`
data_path = "./output_data/UN-2026-03-18"

slice_numbers = [6,10,14]

## Mapper settings
import temporalmapper as tm
mapper_params = {
    "n_neighbors": 500,
    "slice_method": "data",
    "overlap":0.8,
    "kernel":tm.kernels.gaussian,
}
### THE CLUSTERER PARAMS DOMINATE RUNTIME -- MORE CLUSTERS IS EXPONENTIALLY SLOWER ###
clusterer_params = {
    'min_clusters':4,
    'max_layers':3,
    'base_min_cluster_size':50,
    'verbose':False
}

## Toponymy settings
compute_names = False

# Options: 'all-MiniLM-L6-v2' (fast, 384 dim), 'all-mpnet-base-v2' (better quality, 768 dim)
model_name = 'all-MiniLM-L6-v2'

toponymy_object_description = "excerpts from a speech"
toponymy_corpus_description = "United Nations General Debate Transcripts"

toponymy_exemplar_method = "central"
toponymy_keyphrase_method = "information_weighted"
toponymy_subtopic_method = "facility_location"

configuration_dict = {
    'slice_numbers':slice_numbers,
    'mapper_params':mapper_params,
    'model_name':model_name,
    'toponymy_object_description':toponymy_object_description,
    'toponymy_corpus_description':toponymy_corpus_description,
    'toponymy_exemplar_method':toponymy_exemplar_method,
    'toponymy_keyphrase_method':toponymy_keyphrase_method,
    'toponymy_subtopic_method':toponymy_subtopic_method,
    'clusterer_params':clusterer_params,
}

import json
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
def print_ts(txt):
    now = datetime.now(ZoneInfo('US/Eastern'))
    formatted_datetime = now.strftime("%Y-%m-%d %H:%M:%S")
    print(formatted_datetime+": "+txt)

try:
    Path(data_path).mkdir(parents=True, exist_ok=True)
    # can't JSON serialize the kernel func so gotta do its name instead:
    kernel = mapper_params['kernel']
    configuration_dict['mapper_params']['kernel']=mapper_params['kernel'].__name__
    with open(data_path+'/configuration.json', 'w') as f:
        json.dump(configuration_dict, f, indent=4)
    configuration_dict['mapper_params']['kernel'] = kernel
    print_ts(f"Created experimental folder {data_path}, and saved configuation file.")
except Exception as e:
    print('Error:', e)

import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from tqdm.auto import tqdm
import pickle
# Register tqdm with pandas
tqdm.pandas()

# Import the pre-processed dataset (see: UNGDC-DataPrep.ipynb)
df = pd.read_parquet(
    "https://huggingface.co/datasets/kalebr/un-general-debate-corpus-chunked/resolve/main/ungdc-all-chunked.parquet"
)


embedding_vectors = df['embedding'].to_numpy()
reduced_vectors = df['reduced'].to_numpy()
text = df['chunk_text'].to_numpy()
time = df['year'].to_numpy()

# Sort all arrays by time
sorted_idx = np.argsort(time)
time = time[sorted_idx]
text = text[sorted_idx]
reduced_vectors = np.vstack(reduced_vectors[sorted_idx])
embedding_vectors = np.vstack(embedding_vectors[sorted_idx])
reduced_vectors_with_time = np.hstack([reduced_vectors, time.reshape(-1,1)])

2026-03-18 10:38:37: Created experimental folder ./output_data/UN-2026-03-18, and saved configuation file.


In [2]:

## Toponymy Setup
from toponymy import Toponymy, ToponymyClusterer
from toponymy.cluster_layer import ClusterLayerText  
from toponymy.llm_wrappers import AzureAINamer, AnthropicNamer

api_key_file = "cohere.txt"
with open(api_key_file, 'r') as file:
    api_key = file.read().strip()
    
#Initialize Cohere wrapper  
llm=AzureAINamer(
    api_key, 
    endpoint="https://azureaitimcuse5821437469.services.ai.azure.com/models",
    model="Cohere-command-r-08-2024",
)

# Test connection  
print(llm.test_llm_connectivity())

# from pathlib import Path
# if Path("topic_model.pkl").is_file():
#     print("Loading topic model.")
#     with open ('topic_model.pkl', 'rb') as f:
#         topic_model = pickle.load(f)
# else:
#     print("Computing topic model.")
#     topic_model = Toponymy(**toponymy_params)
#     topic_model.fit(text, embedding_vectors, reduced_vectors, **toponymy_fit_params)

Hello! I am Command, a brilliant and sophisticated AI-assistant chatbot. I am here to assist you with your request. 

I will be providing a list of potential topic names in JSON format for your convenience: 

```json
[
  "The Evolution of Artificial Intelligence",
  "Sustainable Energy Solutions for a Greener Future",
  "Mental Health Awareness and Stigma Reduction",
  "The Impact of Social Media on Society",
  "Exploring the Universe: Space Exploration and its Benefits",
  "The Future of Education: Innovative Teaching Methods",
  "Climate Change Mitigation Strategies",
  "Ethical Considerations in


In [3]:

from sentence_transformers import SentenceTransformer
import torch
# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load the embedding model
model = SentenceTransformer(model_name, device=device)
print(f"Loaded model: {model_name}")

from mapperclusterer import MapperClusterer
base_clusterer = ToponymyClusterer(**clusterer_params)

toponymy_params = {
    'llm_wrapper':llm,
    'text_embedding_model':model,
    'object_description':"excerpts from a speech",
    'corpus_description':"United Nations General Debate Transcripts",
    'exemplar_delimiters':["<EXAMPLE_TRANSCRIPT>\n","\n</EXAMPLE_TRANSCRIPT>\n\n"],
}
toponymy_fit_params = {
    'exemplar_method':toponymy_exemplar_method,
    'keyphrase_method':toponymy_keyphrase_method,
    'subtopic_method':toponymy_subtopic_method,
}


Using device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: all-MiniLM-L6-v2


In [4]:
from toponymy.clustering import Clusterer, ClusterLayerText
from temporalmapper import TemporalMapper
from copy import deepcopy 
import networkx as nx
import numpy as np
from scipy.sparse import issparse
from sklearn.utils.validation import check_is_fitted, check_array
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist

print("Checking cluster sizes:")
for n_slices in slice_numbers:
    for density_based in {True, False}:
        mapper_params['n_slices'] = n_slices
        mapper_params['density_based'] = density_based
        clusterer_params['base_min_cluster_size'] = int(120/np.sqrt(n_slices))
        print(f"Computing for N={n_slices}, db={density_based}")
        base_clusterer = ToponymyClusterer(**clusterer_params)
        try:
            clusterer = MapperClusterer(
                base_clusterer=base_clusterer,
                mapper_params=mapper_params
            )
            clusterer.fit(reduced_vectors_with_time, embedding_vectors)
        except Exception as e:
            print(f"Failed to fit db={density_based} slices={n_slices}", e)

Checking cluster sizes:
Computing for N=6, db=False


/work/home/kdrusci/winter2026/dbmapper/tm-dev/temporal-mapper/src/temporalmapper/temporal_mapper.py:128: UserWarning: You have not passed a clusterer, this TemporalMapper cannot be fit.
  warn("You have not passed a clusterer, this TemporalMapper cannot be fit.")


Layers per slice: [3, 3, 3, 3, 3, 3]
Layer 0 n_cluster: [104, 115, 79, 104, 82, 53]
Layer 1 n_cluster: [29, 34, 20, 27, 24, 17]
Layer 2 n_cluster: [10, 11, 9, 9, 9, 7]
Computing for N=6, db=True


/work/home/kdrusci/winter2026/dbmapper/tm-dev/temporal-mapper/src/temporalmapper/temporal_mapper.py:128: UserWarning: You have not passed a clusterer, this TemporalMapper cannot be fit.
  warn("You have not passed a clusterer, this TemporalMapper cannot be fit.")


Layers per slice: [3, 3, 3, 3, 3, 3]
Layer 0 n_cluster: [135, 145, 114, 166, 112, 87]
Layer 1 n_cluster: [40, 37, 36, 44, 29, 24]
Layer 2 n_cluster: [11, 12, 13, 17, 10, 10]
Computing for N=10, db=False


...

Layers per slice: [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
Layer 0 n_cluster: [87, 101, 108, 106, 102, 176, 95, 124, 98, 70]
Layer 1 n_cluster: [29, 31, 32, 30, 31, 53, 26, 32, 29, 22]
Layer 2 n_cluster: [9, 11, 10, 12, 11, 17, 9, 12, 11, 9]
Computing for N=14, db=False


/work/home/kdrusci/winter2026/dbmapper/tm-dev/temporal-mapper/src/temporalmapper/temporal_mapper.py:128: UserWarning: You have not passed a clusterer, this TemporalMapper cannot be fit.
  warn("You have not passed a clusterer, this TemporalMapper cannot be fit.")


Layers per slice: [3, 3, 3, 2, 3, 2, 3, 3, 2, 3, 3, 3, 2, 3]
Layer 0 n_cluster: [58, 55, 99, 21, 98, 24, 63, 121, 27, 91, 24, 83, 22, 57]
Layer 1 n_cluster: [19, 17, 27, 9, 29, 9, 17, 35, 10, 22, 10, 25, 10, 20]
Failed to fit db=False slices=14 list index out of range
Computing for N=14, db=True


/work/home/kdrusci/winter2026/dbmapper/tm-dev/temporal-mapper/src/temporalmapper/temporal_mapper.py:128: UserWarning: You have not passed a clusterer, this TemporalMapper cannot be fit.
  warn("You have not passed a clusterer, this TemporalMapper cannot be fit.")


Layers per slice: [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
Layer 0 n_cluster: [68, 55, 137, 62, 131, 59, 66, 212, 44, 144, 57, 119, 50, 123]
Layer 1 n_cluster: [21, 17, 43, 21, 36, 20, 17, 61, 13, 35, 16, 33, 19, 35]
Layer 2 n_cluster: [7, 6, 14, 9, 12, 9, 8, 18, 5, 13, 8, 12, 9, 13]


In [11]:
from toponymy.serialization import TopicModel

dp = Path(data_path+"/saved")
dp.mkdir(parents=True, exist_ok=True)
print("Computing...")
for n_slices in slice_numbers:
    for density_based in {True, False}:
        fp = Path(data_path+f"/saved/tm_{n_slices}_{density_based}.tm.zip")
        if fp.is_file() and (fp.stat.st_size>1024):
            print(f"Skipping {n_slices}, db={density_based}")
            continue
        mapper_params['n_slices'] = n_slices
        mapper_params['density_based'] = density_based
        clusterer_params['base_min_cluster_size'] = int(120/np.sqrt(n_slices))
        print(f"Computing for N={n_slices}, db={density_based}")
        base_clusterer = ToponymyClusterer(**clusterer_params)
        try:
            clusterer = MapperClusterer(
                base_clusterer=base_clusterer,
                mapper_params=mapper_params
            )
            clusterer.fit(reduced_vectors_with_time, embedding_vectors)
            toponymy_params['clusterer'] = clusterer
            toponymy = Toponymy(**toponymy_params)
            toponymy.fit(
                text,
                embedding_vectors,
                reduced_vectors_with_time,
            )
            tm = TopicModel.from_toponymy(toponymy)
            tm.to_file(str(fp))
        except Exception as e:
            print(f"Failed to fit db={density_based} slices={n_slices}", e)

Computing...
Computing for N=6, db=False


/work/home/kdrusci/winter2026/dbmapper/tm-dev/temporal-mapper/src/temporalmapper/temporal_mapper.py:128: UserWarning: You have not passed a clusterer, this TemporalMapper cannot be fit.
  warn("You have not passed a clusterer, this TemporalMapper cannot be fit.")


Layers per slice: [3, 3, 3, 3, 3, 3]
Layer 0 n_cluster: [104, 115, 79, 104, 82, 53]
Layer 1 n_cluster: [29, 34, 20, 27, 24, 17]
Layer 2 n_cluster: [10, 11, 9, 9, 9, 7]


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Building topic names by layer:   0%|          | 0/3 [00:00<?, ?layer/s]

...

/work/home/kdrusci/winter2026/dbmapper/tm-dev/temporal-mapper/src/temporalmapper/temporal_mapper.py:128: UserWarning: You have not passed a clusterer, this TemporalMapper cannot be fit.
  warn("You have not passed a clusterer, this TemporalMapper cannot be fit.")


Layers per slice: [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
Layer 0 n_cluster: [68, 55, 138, 62, 131, 59, 66, 212, 44, 144, 57, 119, 50, 123]
Layer 1 n_cluster: [21, 17, 43, 21, 36, 20, 17, 61, 13, 35, 16, 33, 19, 35]
Layer 2 n_cluster: [7, 6, 14, 9, 12, 9, 8, 18, 5, 13, 8, 12, 9, 13]


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Building topic names by layer:   0%|          | 0/3 [00:00<?, ?layer/s]

Failed to fit db=True slices=14 operands could not be broadcast together with shapes (0,) (384,) 


## Plotting

In [6]:
from temporalmapper.plotting import squarify_text
import matplotlib.pyplot as plt
def make_cluster_labels(LM):
    cluster_labels = {
        l:{} for l in range(len(LM.mappers))
    }
    
    for layer, mapper in enumerate(LM.mappers):
        for node in mapper.graph.nodes():
            if (mapper.graph.in_degree(node) == 1) and (mapper.graph.out_degree(node)==1):
                 cluster_labels[layer][node] = ""
            else:
                time, cluster = node.split(":")
                time = int(time)
                cluster = int(cluster)
                cluster_labels[layer][node] = squarify_text(LM.topic_models[time].topic_names_[layer][cluster])
    return cluster_labels

def temporal_plot(LM, layer, vertices=None):
    fig, ax = plt.subplots(1,1, figsize=(16,10))
    if vertices is None:
        vertices = LM.mappers[layer].graph.nodes()
    
    fontsize = {2:8,1:6,0:5}
    cluster_labels = make_cluster_labels(LM)
    if layer == 0:
        layout_optimization = 'ordered'
    else:
        layout_optimization = 'ordered'
    LM.mappers[layer].temporal_plot(
        ax=ax,
        cluster_labels = cluster_labels[layer],
        layout_optimization = layout_optimization,
        cluster_label_kwargs = dict(fontsize=fontsize[layer]),
        edge_weight_bounds = (1,5),
        node_size_bounds = (5,50),
        node_scaling = 5,
        vertices = vertices,
    )
    label_times = LM.mappers[layer].checkpoints
    label_text = [int(label) for label in label_times]
    ax.set_xticks(label_times, labels=label_text)
    ax.tick_params(axis='x', labelrotation=90)
    ax.tick_params(bottom=True, labelbottom=True)
    N = LM.mapper_params['N_checkpoints']
    g = LM.mapper_params['overlap']
    k = LM.mapper_params['neighbours']
    title_str = f"UNGDC Mapper Temporal Embedding\n Parameters: N={N}, g={g}, k={k}, Layer={layer} "
    if LM.mapper_params['density_based']:
        title_str += "(density based)"
    else:
        title_str += "(standard mapper)"
    ax.set_title(title_str,  fontsize=18)
    plt.tight_layout()
    return fig

In [7]:
image_path = Path(data_path+"/images")
image_path.mkdir(parents=True, exist_ok=True)

directory_path = Path(data_path+"/pickled") 
pkl_files = list(directory_path.glob("*.pkl"))

for file_path in pkl_files:
    print('plotting from', file_path)
    with open(file_path, 'rb') as f:
        LM = pickle.load(f)
    for layer in range(len(LM.mappers)):
        fig = temporal_plot(LM, layer)
        N = LM.mapper_params['N_checkpoints']
        R = LM.mapper_params['density_based']
        fig.savefig(str(image_path) + "/" + f'tp_N{N}_db{R}_layer{layer}.png')
        fig.show()
        
    


In [8]:
def connected_component(LM, vertex):
    """ Returns nodes in the connected component of `vertex`"""
    layer, time, cluster = vertex.split(":")
    layer = int(layer)
    mapper = LM.mappers[layer]
    subgraph = mapper.vertex_subgraph(time+":"+cluster)
    return subgraph

def down_one_level(LM, vertex):
    """ Returns the nodes connected below 'vertex' in the hierarchy """
    layer, time, cluster = vertex.split(":")
    layer = int(layer)
    time = int(time)
    cluster = int(cluster)
    toponymy = LM.topic_models[time]
    subgraphs = {l:[] for l in range(layer+1)}
    for subvertex in toponymy.cluster_tree_[(layer, cluster)]:
        sublayer, subcluster = subvertex
        for vert in connected_component(LM, f"{sublayer}:{time}:{subcluster}"):
            subgraphs[int(sublayer)].append(vert) 
    subgraphs = {l:np.unique(subgraphs[l]) for l in subgraphs.keys()}
    return subgraphs

In [9]:
for file_path in pkl_files:
    print(file_path)

In [10]:
with open('output_data/UN-2026-03-02-Chronoscope/pickled/layeredmapper_10_dbTrue.pkl', 'rb') as f:
        LM = pickle.load(f)
image_path = Path(data_path+"/images")
layer = 0
t=4
l=1
c=6
name = LM.topic_models[t].topic_names_[l][c]
print("Mapper graph under", name)
subgraphs = down_one_level(LM, f"{l}:{t}:{c}")

fig = temporal_plot(LM, layer, subgraphs[layer])
fig.savefig(str(image_path) + "/" + f'subgraph_{name}.png')
fig.show()

AttributeError: Can't get attribute 'LayeredMapper' on <module '__main__'>